In [3]:
!pip install -q -U google-genai

In [2]:
from google.colab import userdata
userdata.get('sarang_key')

'19'

In [4]:
import time
import random
from google import genai
from google.genai import types
from google.colab import userdata

In [ ]:
api_key=userdata.get("GEMINI_API_KEY") #secure authentication

In [6]:
client=genai.Client(api_key=api_key) #client initailization

In [12]:
prompt_test="Welcome all for today's session on llms"

In [14]:
response=client.models.generate_content(
    model='gemini-3.6-flash',
    contents=prompt_test,
    config=types.GenerateContentConfig(
        temperature=0.7,
        top_p=0.9,
        max_output_tokens=1024,
        candidate_count=1
    )


)

In [15]:
response.text

"Thank you! It’s great to be here. \n\nWhether you're hosting a presentation, running a workshop, or facilitating a discussion, I'm ready to assist! \n\nHow would you like to kick things off today? I can help with:\n\n1. **Setting an Agenda:** Outlining key topics (e.g., Transformer architecture, Prompt Engineering, RAG, Fine-tuning, or Ethics & Safety).\n2. **Answering Audience Questions:** Acting as a subject-matter expert for technical or practical questions.\n3. **Providing Examples & Demos:** Explaining concepts with clear analogies, code snippets, or real-world use cases.\n4. **Drafting an Introduction:** Writing an engaging opening speech or slide text.\n\nWhere shall we begin?"

In [16]:
prompt_test="explain llm"

In [17]:
response=client.models.generate_content(
    model='gemini-3.6-flash',
    contents=prompt_test,
    config=types.GenerateContentConfig(
        temperature=0.7,
        top_p=0.9,
        max_output_tokens=1024,
        candidate_count=1,

    )


)

In [18]:
response.text

"A **Large Language Model (LLM)** is a type of artificial intelligence (AI) designed to understand, process, and generate human language. \n\nExamples of LLMs include OpenAI's **"

In [20]:
def generate_with_backoff(prompt_text, max_retries=5):
    """
    Calls the Gemini API. If the server is busy, it waits an
    exponentially increasing amount of time before trying again.
    """
    for attempt in range(max_retries):
        try:
            # Start timer to measure speed
            start_time = time.time()

            # The API Call
            response = client.models.generate_content(
                model='gemini-3.6-flash', # Fast, cost-effective model
                contents=prompt_text,
                config=types.GenerateContentConfig(
                    temperature=0.7, # Balanced creativity
                    top_p=0.9
                )
            )

            # Stop timer
            end_time = time.time()
            duration = end_time - start_time

            # If successful, return the text and the time it took
            return response.text, duration

        except Exception as e:
            print(f"[Warning] Attempt {attempt + 1} failed. Error: {e}")

            # If this was our final attempt, crash gracefully
            if attempt == max_retries - 1:
                print("Max retries reached. Server is unresponsive.")
                raise e

            # EXPONENTIAL BACKOFF MATH:
            # (2 ^ attempt) + a random decimal (jitter) to prevent synchronized crashes
            sleep_time = (2 ** attempt) + random.uniform(0, 1)
            print(f"Waiting {sleep_time:.2f} seconds before retrying...")

            time.sleep(sleep_time)

# ==========================================
# 3. EXECUTING THE CODE
# ==========================================
# Let's test our robust pipeline!
my_prompt = """
Write a short, professional, but slightly humorous welcome message
for my team members Marten, Tony, and Jaymin, who are joining me on
the new 'Extract Once, Judge Many' pipeline project today.
"""

print("Sending request to Google servers...\n")

# Execute function and unpack variables
result_text, time_taken = generate_with_backoff(my_prompt)

# Print results
print("--- AI RESPONSE ---")
print(result_text)
print("-------------------")
print(f"\n[Metrics] Request completed in {time_taken:.2f} seconds.")

Sending request to Google servers...

--- AI RESPONSE ---
Here is a message option that hits the right balance of professional and lighthearted:

***

**Subject:** Welcome to the team – Extract Once, Judge Many!

Welcome to the project, Marten, Tony, and Jaymin! 

Today we officially kick off the **Extract Once, Judge Many** pipeline. As the name implies, our goal is peak efficiency—and a healthy amount of relentless judging (strictly limited to our pipeline outputs, of course). 

I’m thrilled to have the three of you on board for this. Let’s dive in and build something great!

***

**Alternative (Shorter version for Slack/Teams):**

Welcome to the team, @Marten, @Tony, and @Jaymin! Today marks Day 1 of the **Extract Once, Judge Many** pipeline project. Don't worry, no courtroom robes are required, but we *will* be doing some heavy evaluation. Excited to get started with you all—let's build something great!
-------------------

[Metrics] Request completed in 46.23 seconds.
